# CreatorPulse AI

A Python analytics notebook for creator income reporting, unusual-payment detection, revenue forecasting, and chart-based insights.

**Data note:** The notebook generates synthetic example data. Exchange rates are illustrative, not live financial rates. Replace them with verified data before using this workflow for real decisions.

In [ ]:
# CreatorPulse AI: Creator Income Analytics, Anomaly Detection, and Revenue Forecasting
# Uses synthetic sample data. Replace it with real data before making business decisions.

import sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")

for module, package in {"numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","sklearn":"scikit-learn"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import mean_absolute_error

SEED = 42
rng = np.random.default_rng(SEED)
n_days = 365
dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=n_days, freq="D")
platforms = ["YouTube", "Instagram", "TikTok", "Freelance", "Brand Partnership"]
currencies = ["USD", "EUR", "GBP", "INR"]
rates_to_inr = {"USD": 83.0, "EUR": 90.0, "GBP": 105.0, "INR": 1.0}  # Illustrative rates only
platform_probs = [0.28, 0.22, 0.14, 0.20, 0.16]

df = pd.DataFrame({
    "date": dates,
    "platform": rng.choice(platforms, n_days, p=platform_probs),
    "currency": rng.choice(currencies, n_days, p=[0.55, 0.18, 0.12, 0.15]),
    "views": rng.integers(500, 250_000, n_days),
    "engagement_rate": np.clip(rng.normal(0.045, 0.018, n_days), 0.002, 0.18),
    "base_payment": rng.lognormal(4.0, 0.75, n_days),
})
multipliers = df["platform"].map({
    "YouTube": 1.4, "Instagram": 1.1, "TikTok": 0.85,
    "Freelance": 1.3, "Brand Partnership": 1.8
}).astype(float)
df["amount_original"] = (
    df["base_payment"] * multipliers * (0.65 + df["views"] / 100_000)
    * (0.7 + df["engagement_rate"] * 5)
).round(2)

# Add a few synthetic unusual values for demonstration.
odd_rows = rng.choice(df.index, size=6, replace=False)
df.loc[odd_rows, "amount_original"] *= rng.choice([0.08, 5.5, 8.0], size=6)
df["amount_original"] = df["amount_original"].round(2)
df["exchange_rate_to_inr"] = df["currency"].map(rates_to_inr)
df["amount_inr"] = (df["amount_original"] * df["exchange_rate_to_inr"]).round(2)
df["estimated_cost_inr"] = (df["amount_inr"] * rng.uniform(0.05, 0.22, n_days)).round(2)
df["estimated_profit_inr"] = (df["amount_inr"] - df["estimated_cost_inr"]).round(2)
df["month"] = df["date"].dt.to_period("M").astype(str)

print("DATA QUALITY REPORT")
print("Rows:", len(df), "| Duplicate rows:", int(df.duplicated().sum()))
print("Missing values:", int(df.isna().sum().sum()))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
display(df.head(10))

# Key performance indicators
revenue = df["amount_inr"].sum()
costs = df["estimated_cost_inr"].sum()
profit = df["estimated_profit_inr"].sum()
margin = profit / revenue * 100 if revenue else 0
kpis = pd.DataFrame({
    "Metric": ["Total revenue (INR)", "Estimated costs (INR)", "Estimated profit (INR)",
               "Average payment (INR)", "Estimated profit margin (%)", "Total views",
               "Average engagement rate (%)"],
    "Value": [round(revenue, 2), round(costs, 2), round(profit, 2),
              round(df["amount_inr"].mean(), 2), round(margin, 2),
              int(df["views"].sum()), round(df["engagement_rate"].mean() * 100, 2)]
})
print("KEY PERFORMANCE INDICATORS")
display(kpis)

platform_summary = df.groupby("platform", as_index=False).agg(
    payments=("amount_inr", "count"),
    revenue_inr=("amount_inr", "sum"),
    profit_inr=("estimated_profit_inr", "sum"),
    average_views=("views", "mean"),
    average_engagement=("engagement_rate", "mean")
).sort_values("revenue_inr", ascending=False)
platform_summary["revenue_share_pct"] = (
    platform_summary["revenue_inr"] / platform_summary["revenue_inr"].sum() * 100
).round(2)
platform_summary["average_views"] = platform_summary["average_views"].round().astype(int)
platform_summary["average_engagement_pct"] = (platform_summary.pop("average_engagement") * 100).round(2)

monthly_summary = df.groupby("month", as_index=False).agg(
    revenue_inr=("amount_inr", "sum"),
    profit_inr=("estimated_profit_inr", "sum"),
    payments=("amount_inr", "count"),
    views=("views", "sum")
)
currency_summary = df.groupby("currency", as_index=False).agg(
    payment_count=("amount_original", "count"),
    original_currency_total=("amount_original", "sum"),
    revenue_inr=("amount_inr", "sum")
).sort_values("revenue_inr", ascending=False)

print("PLATFORM PERFORMANCE")
display(platform_summary)
print("MONTHLY PERFORMANCE")
display(monthly_summary.tail(12))
print("CURRENCY BREAKDOWN")
display(currency_summary)

# Anomaly detection: flags unusual patterns; it does not prove fraud or error.
detector = IsolationForest(contamination=0.025, random_state=SEED)
df["anomaly_label"] = detector.fit_predict(np.log1p(df[["amount_inr", "views"]].clip(lower=0)))
df["is_unusual_payment"] = df["anomaly_label"].eq(-1)
unusual_payments = df.loc[df["is_unusual_payment"],
    ["date", "platform", "currency", "amount_original", "amount_inr", "views"]
].sort_values("amount_inr", ascending=False)
print("UNUSUAL PAYMENTS FOR REVIEW")
display(unusual_payments)

# Revenue forecasting with a chronological holdout.
daily = df.groupby("date", as_index=False).agg(daily_revenue_inr=("amount_inr", "sum")).sort_values("date")
daily["day_index"] = np.arange(len(daily))
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["day_of_year"] = daily["date"].dt.dayofyear
daily["rolling_mean_7"] = daily["daily_revenue_inr"].shift(1).rolling(7, min_periods=1).mean()
daily["rolling_mean_30"] = daily["daily_revenue_inr"].shift(1).rolling(30, min_periods=1).mean()
daily["lag_1"] = daily["daily_revenue_inr"].shift(1)
daily["lag_7"] = daily["daily_revenue_inr"].shift(7)
daily["lag_30"] = daily["daily_revenue_inr"].shift(30)
features = ["day_index", "day_of_week", "day_of_year", "rolling_mean_7",
            "rolling_mean_30", "lag_1", "lag_7", "lag_30"]
model_data = daily.dropna(subset=features + ["daily_revenue_inr"]).copy()
forecast_df = pd.DataFrame(columns=["date", "predicted_revenue_inr"])
mae = np.nan

if len(model_data) >= 60:
    split = int(len(model_data) * 0.8)
    train, test = model_data.iloc[:split], model_data.iloc[split:]
    model = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=SEED, n_jobs=-1)
    model.fit(train[features], train["daily_revenue_inr"])
    predictions = model.predict(test[features])
    mae = mean_absolute_error(test["daily_revenue_inr"], predictions)
    print(f"Chronological holdout MAE: INR {mae:,.2f}")

    history = daily[["date", "daily_revenue_inr"]].copy().reset_index(drop=True)
    future = []
    for _ in range(30):
        next_date = history["date"].iloc[-1] + pd.Timedelta(days=1)
        vals = history["daily_revenue_inr"].to_numpy()
        row = {
            "day_index": len(history), "day_of_week": next_date.dayofweek,
            "day_of_year": next_date.dayofyear,
            "rolling_mean_7": float(np.mean(vals[-7:])),
            "rolling_mean_30": float(np.mean(vals[-30:])),
            "lag_1": float(vals[-1]), "lag_7": float(vals[-7]), "lag_30": float(vals[-30])
        }
        pred = max(0.0, float(model.predict(pd.DataFrame([row])[features])[0]))
        future.append({"date": next_date, "predicted_revenue_inr": pred})
        history = pd.concat([history, pd.DataFrame([{"date": next_date, "daily_revenue_inr": pred}])],
                            ignore_index=True)
    forecast_df = pd.DataFrame(future)
    print("30-day experimental forecast")
    display(forecast_df.head(10))
    print("Forecast total (INR):", round(forecast_df["predicted_revenue_inr"].sum(), 2))

# Visualizations
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["month"], monthly_summary["revenue_inr"], marker="o")
plt.title("Monthly Creator Revenue"); plt.xlabel("Month"); plt.ylabel("Revenue (INR)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 5))
plt.bar(platform_summary["platform"], platform_summary["revenue_inr"])
plt.title("Revenue by Platform"); plt.xlabel("Platform"); plt.ylabel("Revenue (INR)")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 5))
plt.bar(platform_summary["platform"], platform_summary["profit_inr"])
plt.title("Estimated Profit by Platform"); plt.xlabel("Platform"); plt.ylabel("Estimated Profit (INR)")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(df["views"], df["amount_inr"], alpha=0.65)
plt.title("Payment Amount vs Views"); plt.xlabel("Views"); plt.ylabel("Payment (INR)")
plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df["amount_inr"], bins=30)
plt.title("Distribution of Payments"); plt.xlabel("Payment (INR)"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 5))
plt.plot(daily["date"], daily["daily_revenue_inr"], label="Historical revenue")
if not forecast_df.empty:
    plt.plot(forecast_df["date"], forecast_df["predicted_revenue_inr"], label="Forecast")
plt.title("Daily Revenue and 30-Day Forecast"); plt.xlabel("Date"); plt.ylabel("Revenue (INR)")
plt.legend(); plt.tight_layout(); plt.show()

# Summary insights
best_platform = platform_summary.iloc[0]
best_month = monthly_summary.loc[monthly_summary["revenue_inr"].idxmax()]
print("AUTOMATED INSIGHTS")
print(f"Highest-revenue platform: {best_platform['platform']} (INR {best_platform['revenue_inr']:,.2f})")
print(f"Highest-revenue month: {best_month['month']} (INR {best_month['revenue_inr']:,.2f})")
print(f"Estimated overall profit margin: {margin:.2f}%")
print(f"Unusual payments flagged for review: {int(df['is_unusual_payment'].sum())}")
if not np.isnan(mae):
    print(f"Forecast holdout MAE: INR {mae:,.2f}")
print("Note: data is synthetic; exchange rates are illustrative; forecasts are experimental.")

# Export tables to CSV files in the current notebook folder.
exports = {
    "creatorpulse_transactions.csv": df,
    "creatorpulse_kpis.csv": kpis,
    "creatorpulse_platform_summary.csv": platform_summary,
    "creatorpulse_monthly_summary.csv": monthly_summary,
    "creatorpulse_currency_summary.csv": currency_summary,
    "creatorpulse_unusual_payments.csv": unusual_payments,
    "creatorpulse_30_day_forecast.csv": forecast_df
}
for filename, table in exports.items():
    table.to_csv(filename, index=False)
print("CSV exports created:")
for filename in exports:
    print(" -", filename)
